In [1]:
import argparse
import functools
import gc
import itertools
import logging
import math
import os
from distutils.util import strtobool
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
import transformers
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import (
    DistributedDataParallelKwargs,
    ProjectConfiguration,
    set_seed,
)
from huggingface_hub import create_repo, upload_folder
from huggingface_hub.utils import insecure_hashlib
from packaging import version
from PIL import Image
from PIL.ImageOps import exif_transpose
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoTokenizer, PretrainedConfig

import diffusers
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
    StableDiffusionXLPipeline,
)

from diffusers.loaders import LoraLoaderMixin
from diffusers.optimization import get_scheduler
from diffusers.utils import check_min_version, is_wandb_available
from diffusers.utils.import_utils import is_xformers_available

from unziplora_unet.unziplora_linear_layer import UnZipLoRALinearLayer
from unziplora_unet.pipeline_stable_diffusion_xl import StableDiffusionXLUnZipLoRAPipeline
from unziplora_unet.unet_2d_condition import UNet2DConditionModel
from unziplora_unet.utils import *

In [2]:
unet = UNet2DConditionModel.from_pretrained(
    '/home/xzh/xzh/pretrained/sd_xl_base_1.0', subfolder='unet'
)
unet

UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (add_time_proj): Timesteps()
  (add_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=2816, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): DownBlock2D(
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(
          (norm1): GroupNorm(32, 320, eps=1e-05, affine=True)
          (conv1): Conv2d(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=320, bias=True)
          (norm2): GroupNorm(32, 320, eps=1e-05, affine=True)
          (dropout): Dropout(p=

In [13]:
for attn_processor_name, attn_processor in unet.attn_processors.items():
    attn_module = unet
    print(attn_processor_name , attn_processor)

down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f9faeb75110>
down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f9fc0d85710>
down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f9faeb77ad0>
down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f9faeb74e90>
down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f9faeb54310>
down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f9faeb90a10>
down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f9faeb91e50>